# TASK 3 - Digital Twin

For this task, we were asked to build a digital twin from an image of a chessboard. Therefore, we had to:
- Identify the position of each piece relative to the chessboard squares
- Identify the type of each piece

The following pipeline, describes the process created, and developed in this notebook:

![Pipeline](notebook_imgs/pipeline.png)

### Import Libraries and Setup

In this section, we import all the necessary libraries for image processing, data handling, and model training. We also check the Ultralytics YOLO version and set a random seed for reproducibility.

In [ ]:
pip install -r requirements.txt

In [ ]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch
import ultralytics
import random

# Check Ultralytics version
print(f"Ultralytics version: {ultralytics.__version__}")
if ultralytics.__version__ < '8.3.146':
    print("Warning: Ultralytics version is outdated. Please update with 'pip install -U ultralytics'")

# Set random seed for reproducibility
random.seed(42)

### Define YOLO Annotation Function

This function converts the dataset annotations from COCO format to YOLO format for a given data split (train/val/test). It creates YOLO annotation `.txt` files and copies the corresponding images to the output directory.

In [ ]:
def create_yolo_annotations(root_dir, output_dir, partition='train'):
    """
    Convert annotations.json to YOLO format for a given partition (train/val/test).
    Creates .txt files with class_id and normalized bbox coordinates.
    Returns list of image paths for the partition.
    """
    # Load annotations
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return [], {}
    anns = json.load(open(anns_file))
    categories = {c['id']: c['name'] for c in anns['categories']}
    image_info = {img['id']: img for img in anns['images']}

    # Check if partition exists
    if partition not in anns['splits']['chessred2k']:
        print(f"Error: Partition '{partition}' not found in annotations.json")
        return [], categories

    # Get split IDs
    split_ids = np.asarray(anns['splits']['chessred2k'][partition]['image_ids']).astype(int)
    image_ids = [img['id'] for img in anns['images'] if img['id'] in split_ids]
    image_paths = []
    images_processed = 0
    images_skipped = 0

    # Create output directory for labels
    label_dir = os.path.join(output_dir, partition, 'labels')
    image_dir = os.path.join(output_dir, partition, 'images')
    os.makedirs(label_dir, exist_ok=True)
    os.makedirs(image_dir, exist_ok=True)

    for image_id in image_ids:
        img_info = image_info[image_id]
        file_name = img_info['path']
        img_path = os.path.join(root_dir, file_name)
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            images_skipped += 1
            continue
        width, height = img_info['width'], img_info['height']

        # Copy image to output directory
        output_img_path = os.path.join(image_dir, os.path.basename(file_name))
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not load image {img_path}")
            images_skipped += 1
            continue
        cv2.imwrite(output_img_path, img)
        image_paths.append(os.path.join(partition, 'images', os.path.basename(file_name)))
        images_processed += 1

        # Create YOLO annotation file
        label_path = os.path.join(label_dir, os.path.splitext(os.path.basename(file_name))[0] + '.txt')
        with open(label_path, 'w') as f:
            for piece in anns['annotations']['pieces']:
                if piece['image_id'] == image_id and 'bbox' in piece:
                    x, y, w, h = piece['bbox']
                    class_id = piece['category_id']
                    # Normalize coordinates: center_x, center_y, width, height
                    center_x = (x + w / 2) / width
                    center_y = (y + h / 2) / height
                    norm_w = w / width
                    norm_h = h / height
                    # Ensure coordinates are within [0, 1]
                    if 0 <= center_x <= 1 and 0 <= center_y <= 1 and norm_w > 0 and norm_h > 0:
                        f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {norm_w:.6f} {norm_h:.6f}\n")
                    else:
                        print(f"Warning: Invalid bbox for image_id {image_id}: {piece['bbox']}")

    print(f"Processed {images_processed} images for {partition} split, skipped {images_skipped}")
    return image_paths, categories

#### Define Data YAML Creation Function

This function creates a `data.yaml` file for YOLO training, specifying the paths to the train, validation, and test image directories, as well as the class names.

In [ ]:
def create_data_yaml(output_dir, categories):
    """
    Create data.yaml file for YOLOv11 training with absolute paths.
    """
    # Get absolute paths
    train_dir = os.path.abspath(os.path.join(output_dir, 'train', 'images'))
    val_dir = os.path.abspath(os.path.join(output_dir, 'val', 'images'))
    test_dir = os.path.abspath(os.path.join(output_dir, 'test', 'images'))

    yaml_content = f"""
train: {train_dir}
val: {val_dir}
test: {test_dir}
nc: {len(categories)}
names: {list(categories.values())}
"""
    yaml_path = os.path.join(output_dir, 'data.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    # Print data.yaml contents for debugging
    print(f"data.yaml contents:\n{yaml_content}")
    return yaml_path

### Training Pipeline

This section prepares the dataset and configuration files for YOLO training. It converts the dataset annotations to YOLO format, creates the necessary directory structure, and generates the `data.yaml` file. Finally, it loads a pretrained YOLO model and starts the training process using the specified parameters.

In [ ]:
root_dir = ''  # Update with your dataset path
output_dir = 'yolo_chess_dataset'  # Output directory for YOLO format
model_name = 'yolo11n.pt'  # Pretrained YOLOv11 model
img_size = 640  # Training image size
epochs = 50  # Number of training epochs
batch_size = 16  # Batch size (adjust based on GPU memory)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create YOLO annotations
print("Converting annotations for training set...")
train_paths, categories = create_yolo_annotations(root_dir, output_dir, 'train')
print("Converting annotations for validation set...")
val_paths, _ = create_yolo_annotations(root_dir, output_dir, 'val')
print("Converting annotations for test set...")
test_paths, _ = create_yolo_annotations(root_dir, output_dir, 'test')

Just run the following function if you want to retrain the Yolo model.

In [ ]:
# Debug: Check available splits
anns_file = os.path.join(root_dir, 'annotations.json')
if os.path.exists(anns_file):
    anns = json.load(open(anns_file))
    print(f"Available splits in annotations.json: {list(anns['splits']['chessred2k'].keys())}")
else:
    print(f"Error: Annotations file {anns_file} not found")

# Check for empty splits
if not train_paths:
    print("Error: No training images found. Cannot proceed with training.")
elif not val_paths:
    print("Warning: No validation images found. Using training set for validation.")
    val_paths = train_paths  # Fallback to train set for validation

# Create data.yaml
yaml_path = create_data_yaml(output_dir, categories)
print(f"Created data.yaml at {yaml_path}")

# Verify directories
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(output_dir, split, 'images')
    if os.path.exists(img_dir) and len(os.listdir(img_dir)) > 0:
        print(f"{split} images directory: {img_dir} contains {len(os.listdir(img_dir))} images")
    else:
        print(f"Warning: {split} images directory {img_dir} is empty or does not exist")

# Load and train YOLOv11 model
print(f"Loading pretrained model {model_name} on {device}")
model = YOLO(model_name)
print("Starting training...")
results = model.train(
    data=yaml_path,
    imgsz=img_size,
    epochs=epochs,
    batch=batch_size,
    name='chess_piece_detection',
    plots=True,
    device=device,
    patience=20,  # Early stopping after 20 epochs without improvement
    save=True,  # Save checkpoints
    save_period=10  # Save every 10 epochs
)

### Test Model on Sample Image

After training, we test the YOLO model on a sample image from the test set. This cell loads the trained model, runs inference on a test image, and visualizes the predicted bounding boxes and class labels for the detected chess pieces.

In [ ]:
test_image_path = os.path.join(output_dir, test_paths[0])
if not os.path.exists(test_image_path):
    print(f"Error: Test image {test_image_path} not found")
else:
    print("Testing model on a sample image...")
    model_path = 'runs/detect/chess_piece_detection/weights/best.pt'
    model = YOLO(model_path)
    results = model.predict(test_image_path, save=False, conf=0.5)

    # Load and display image with predictions
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"Error: Could not load test image {test_image_path}")
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = box.conf.item()
                cls = int(box.cls.item())
                label = f"{categories[cls]} {conf:.2f}"
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, label, (x1, max(y1 - 10, 10)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

        plt.imshow(img)
        plt.axis('off')
        plt.title(f"YOLOv11 Predictions on {os.path.basename(test_image_path)}")
        plt.show()

        print(f"Test image: {test_image_path}")
        print(f"Detected {len(result.boxes)} objects:")
        for box in result.boxes:
            cls = int(box.cls.item())
            conf = box.conf.item()
            print(f" - {categories[cls]} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

### Inference (Utility Functions)

This section defines utility functions for running inference on images using the trained YOLO model. These functions help load class names, retrieve test images, and visualize/save the detection results for further analysis.

In [ ]:
def load_categories(root_dir):
    """
    Load category names from annotations.json.
    """
    anns_file = os.path.join(root_dir, 'annotations.json')
    if not os.path.exists(anns_file):
        print(f"Error: Annotations file {anns_file} not found")
        return {}
    with open(anns_file, 'r') as f:
        anns = json.load(f)
    return {c['id']: c['name'] for c in anns['categories']}

def get_test_image_paths(output_dir):
    """
    Get list of image paths in the test set.
    """
    test_image_dir = os.path.join(output_dir, 'test', 'images')
    if not os.path.exists(test_image_dir):
        print(f"Error: Test images directory {test_image_dir} not found")
        return []
    return [os.path.join(test_image_dir, img) for img in os.listdir(test_image_dir) if img.endswith(('.jpg', '.jpeg', '.png'))]

def test_on_image(model, image_path, categories, output_dir, conf_threshold=0.5):
    """
    Run inference on a single image, display results, and save annotated image.
    """
    if not os.path.exists(image_path):
        print(f"Error: Image {image_path} not found")
        return

    results = model.predict(image_path, save=False, conf=conf_threshold)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image {image_path}")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_annotated = img.copy()

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f"YOLOv11 Predictions on {os.path.basename(image_path)}")
    plt.show()

    predictions_dir = os.path.join(output_dir, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)
    output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(image_path)}")
    cv2.imwrite(output_image_path, img_annotated)
    print(f"Saved annotated image to: {output_image_path}")

    print(f"Test image: {image_path}")
    print(f"Detected {len(result.boxes)} objects:")
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = box.conf.item()
        print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

### Run Inference on Test Image

Here, we use the trained YOLO model to run inference on several test images. The results are visualized side-by-side with the original images, and the annotated images are saved for inspection. This step demonstrates the model's performance on unseen data.

In [ ]:
root_dir = ''  # Update with your dataset path
output_dir = 'yolo_chess_dataset'
model_path = 'runs/detect/chess_piece_detection/weights/best.pt'
conf_threshold = 0.7

# Load categories and test images
categories = load_categories(root_dir)
test_image_paths = get_test_image_paths(output_dir)


# Limit to 3 test images or fewer if available
test_image_paths = test_image_paths[:3]
print(f"Loading trained model from {model_path}")
model = YOLO(model_path)

for test_image_path in test_image_paths:
    if not os.path.exists(test_image_path):
        print(f"Error: Image {test_image_path} not found")
        continue

    print(f"Testing on image: {test_image_path}")
    
    # Run inference
    results = model.predict(test_image_path, save=False, conf=conf_threshold)
    
    # Load original image
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"Error: Could not load image {test_image_path}")
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_annotated = img.copy()  # For annotations (BGR)

    # Draw bounding boxes and larger labels
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            # Draw rectangle (BGR for OpenCV, red)
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            # Add larger label
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    # Convert annotated image to RGB for display
    img_annotated_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
    
    # Display side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    ax1.imshow(img_rgb)
    ax1.set_title("Original Image")
    ax1.axis('off')
    ax2.imshow(img_annotated_rgb)
    ax2.set_title(f"Predictions on {os.path.basename(test_image_path)}")
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

    # Save annotated image
    predictions_dir = os.path.join(output_dir, 'predictions')
    os.makedirs(predictions_dir, exist_ok=True)
    output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(test_image_path)}")
    cv2.imwrite(output_image_path, img_annotated)
    print(f"Saved annotated image to: {output_image_path}")

    # Print results
    print(f"Test image: {test_image_path}")
    print(f"Detected {len(result.boxes)} objects:")
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = box.conf.item()
        print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")

In [ ]:
def run_inference_by_id(image_id, root_dir='', output_dir='yolo_chess_dataset', model_path='runs/detect/chess_piece_detection/weights/best.pt', conf_threshold=0.7, debug=False):

    # Load categories
    categories = load_categories(root_dir)
    
    # Construct full image path
    test_image_path = os.path.join(output_dir, 'test', 'images', f"{image_id}.jpg")
    if not os.path.exists(test_image_path):
        print(f"Error: Image {test_image_path} not found")
        return None, None, None
    
    print(f"Running inference on image: {test_image_path}")
    
    # Load model
    model = YOLO(model_path)
    
    # Run inference
    results = model.predict(test_image_path, save=False, conf=conf_threshold)
    
    img_rgb, img_annotated_rgb = None, None

    
        
    # Load original image
    img = cv2.imread(test_image_path)
    if img is None:
        print(f"Error: Could not load image {test_image_path}")
        return None, None, None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_annotated = img.copy()  # For annotations (BGR)

    # Draw bounding boxes and larger labels
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = box.conf.item()
            cls = int(box.cls.item())
            label = f"{categories.get(cls, 'unknown')} {conf:.2f}"
            # Draw rectangle (BGR for OpenCV, red)
            cv2.rectangle(img_annotated, (x1, y1), (x2, y2), (0, 0, 255), 2)
            # Add larger label
            cv2.putText(img_annotated, label, (x1, max(y1 - 10, 10)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    # Convert annotated image to RGB for display
    img_annotated_rgb = cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB)
        
    if debug:
        # Display side-by-side comparison
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        ax1.imshow(img_rgb)
        ax1.set_title("Original Image")
        ax1.axis('off')
        ax2.imshow(img_annotated_rgb)
        ax2.set_title(f"Predictions on {os.path.basename(test_image_path)}")
        ax2.axis('off')
        plt.tight_layout()
        plt.show()

        # Save annotated image
        predictions_dir = os.path.join(output_dir, 'predictions')
        os.makedirs(predictions_dir, exist_ok=True)
        output_image_path = os.path.join(predictions_dir, f"pred_{os.path.basename(test_image_path)}")
        cv2.imwrite(output_image_path, img_annotated)
        print(f"Saved annotated image to: {output_image_path}")

        # Print results
        print(f"Detected {len(result.boxes)} objects:")
        for box in result.boxes:
            cls = int(box.cls.item())
            conf = box.conf.item()
            print(f" - {categories.get(cls, 'unknown')} at {box.xyxy[0].tolist()} with confidence {conf:.2f}")
    
    return results, img_rgb, img_annotated_rgb


# Detect Corners



Effectively detecting the chessboard corners is a vital step for successfully completing task 3. Initially, we implemented a **regression model that predicted the coordinates of 4 points** (x and y positions, so 8 values in total). Although the results were decent, it led to unsable results too often, which compromised our entire pipeline. Taking inspiration from approaches used in fields such as medical imaging, we built a **regression model that predicts 4 heatmaps**. We then extract the point of highest intensity from each of these maps, which provided us much better  results overall.

### Setup Preparation

In [ ]:
import numpy as np 
import os
import torch
from torch.utils.data import Dataset, DataLoader
import json
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
import copy
import cv2

In [ ]:
def heatmaps_to_points(heatmaps):
    """
    Calculates the points of highest intensity from the heatmaps 
    """
    num_corners, _, W = heatmaps.shape
    points = []

    for i in range(num_corners):
        heatmap = heatmaps[i]
        _, idx = torch.max(heatmap.view(-1), dim=0)
        y, x = divmod(idx.item(), W)
        points.append((x, y))
    
    return points

def points_to_image_coordinates(points, origin_size, target_size):
    """
    Scales the points from the heatmap to the scale of the target image 
    """
    img_w, img_h = target_size
    scale_x = img_w / origin_size
    scale_y = img_h / origin_size
    
    image_points = [
        (x * scale_x, y * scale_y) for (x, y) in points
    ]
    
    return image_points

def make_gaussian_heatmap(size, center, sigma=2):
    """
    Creates a heatmap centered at a given point. Used to generate ground truth heatmaps for training.
    """
    x = np.arange(0, size[1], 1, np.float32)
    y = np.arange(0, size[0], 1, np.float32)
    y = y[:, np.newaxis]

    x0, y0 = center
    heatmap = np.exp(- ((x - x0) ** 2 + (y - y0) ** 2) / (2 * sigma ** 2))
    return heatmap

The **CornerDataset** class loads each image and its annotated corners, then generates a Gaussian heatmap for each corner. These heatmaps serve as the training targets, allowing the model to learn to localize each corner as a region of high intensity.

In [ ]:
class CornerDataset(Dataset):
    def __init__(self, annotations, image_dir, split='train', heatmap_size=56, sigma=2):
        with open(annotations, 'r') as f:
            data = json.load(f)
            all_image_annotations = data["images"]
            corners_list = data["annotations"]["corners"]
        
        self.image_dir = image_dir
        self.corners_annotations = {
            corner['image_id']: corner['corners'] for corner in corners_list
        }
        
        self.image_annotations = []
        for item in all_image_annotations:
            img_path = os.path.join(image_dir, item["path"])
            if os.path.isfile(img_path) and (item["id"] in self.corners_annotations) and (item["id"] in data["splits"]["chessred2k"][split]["image_ids"]):
                self.image_annotations.append(item)

        print(f"Loaded {len(self.image_annotations)} images for {split} split")
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])
        
        self.heatmap_size = heatmap_size
        self.sigma = sigma

    def __len__(self):
        return len(self.image_annotations)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_annotations[idx]['path'])
        image = Image.open(img_path).convert("RGB")
        original_width, original_height = image.size
        image = self.transform(image)
        
        img_id = self.image_annotations[idx]['id']
        corners_dict = self.corners_annotations[img_id]
        ordered_keys = ['top_left', 'top_right', 'bottom_right', 'bottom_left']
        corners = [corners_dict[key] for key in ordered_keys]

        normalized_corners = [
            [x / original_width, y / original_height] for (x, y) in corners
        ]
        
        heatmaps = []
        for (x_norm, y_norm) in normalized_corners:
            x_hm = x_norm * self.heatmap_size
            y_hm = y_norm * self.heatmap_size
            heatmap = make_gaussian_heatmap(
                (self.heatmap_size, self.heatmap_size),
                (x_hm, y_hm),
                sigma=self.sigma
            )
            heatmaps.append(heatmap)

        heatmaps = np.stack(heatmaps)  
        heatmaps = torch.tensor(heatmaps, dtype=torch.float32)
        
        return image, img_path, heatmaps

This class defines our model, which is based on a pre-trained ResNet-34. We experimented with different architectures, but ResNet-34 provided the best balance between overall performance and training time. The final layers of the model were modified to predict heatmaps. These layers **upsample the feature maps produced by the ResNet backbone**, gradually increasing their spatial resolution. This allows the model to output a set of heatmaps (one for each corner), where each heatmap highlights the predicted location of a specific chessboard corner.

In [ ]:
class ResNetHeatmap(nn.Module):
    def __init__(self, num_corners=4):
        super(ResNetHeatmap, self).__init__()
        
        resnet = models.resnet34(weights='DEFAULT')
        self.features = nn.Sequential(*list(resnet.children())[:-2])  
        
        self.head = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2), nn.ReLU(),  
            nn.ConvTranspose2d(256, 128, 2, stride=2), nn.ReLU(),  
            nn.ConvTranspose2d(128, num_corners, 2, stride=2)       
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNetHeatmap().to(device)

The dataset is divided into 3 smaller datasets, following what was studied in class:
- Training Dataset
- Eval Dataset
- Test Dataset

This division is based on the *annotations.json* file, that already makes this division for us.

In [ ]:
train_dataset = CornerDataset("annotations.json", "", split='train')
val_dataset = CornerDataset("annotations.json", "", split='val')
test_dataset = CornerDataset("annotations.json", "", split='test')

batch_size = 32

train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, collate_fn=lambda x: x, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size, collate_fn=lambda x: x, num_workers=0, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size, collate_fn=lambda x: x, num_workers=0, pin_memory=True)

### Model Training

For training the model, several techniques discussed in class were used to optimize its performance, including:
- Early stopping
- Initially training only the head of the model, followed by training the entire model

The model was not trained in this notebook, as it is very time consuming, but it was trained in kaggle and the outputs can be seen [here](https://www.kaggle.com/code/jaimefrancisco123/detectcorners-3).

In [ ]:
def one_epoch(model, optimizer, dataloader, loss_fn, is_training):
    model.train() if is_training else model.eval()
    avg_loss = 0.0

    for batch in dataloader:
        images = torch.stack([b[0] for b in batch]).to(device)
        targets = torch.stack([b[1] for b in batch]).to(device)

        with torch.set_grad_enabled(is_training):
            preds = model(images)  
            loss = loss_fn(preds, targets)  
            if is_training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        avg_loss += loss.item() / len(dataloader)
    return avg_loss

The loss, is the condition used to do the early stopping. A patience of 5 was chosen as it provided a good balance between results and training time, as well. A patience of 3, for example, failed too early on.

In [ ]:
def train_model(optimizer, criterion):
    """
    Trains the model, with early stopping technique. Returns the best state of the model it finds. 
    """
    epochs = 150
    patience = 5
    best_val_metric = float('inf')
    best_epoch = 0

    for epoch in range(epochs):
        train_loss = one_epoch(model, optimizer, train_dataloader, criterion, is_training=True)
        val_loss = one_epoch(model, optimizer, val_dataloader, criterion, is_training=False)
        print(f"Epoch {epoch+1}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")    

        if val_loss < best_val_metric:
            best_val_metric = val_loss
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:
            print(f"Early stopping at epoch {epoch+1}. Best validation MPE: {best_val_metric:.4f}")
            break
    return best_model_state

Initially, we freeze the entire model, except the final layers, and train it.

In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = nn.MSELoss()
best_model_state = train_model(optimizer, criterion)

model.load_state_dict(best_model_state)

Then, we unfreeze these layers, and train the model with a lower learning rate.

In [ ]:
for param in model.features.parameters():
    param.requires_grad = True
    
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
best_model_state = train_model(optimizer, criterion)
model.load_state_dict(best_model_state)

### Results 

To improve, the workflow pipeline, the model was saved in *modelo.pth*.

In [ ]:
if os.path.exists('modelo.pth'):
    print("Loading saved model...")
    model.load_state_dict(torch.load('modelo.pth', map_location=torch.device('cpu') ))
else:
    print("No saved model found.")

To better understand how our model predicts the chessboard corners, we visualize both the ground truth and predicted heatmaps for each corner. This side-by-side comparison helps illustrate where the model is focusing its attention and how closely its predictions match the actual corner locations.

In [ ]:
import matplotlib.cm as cm

# Assume you have: sample_image, pred_heatmaps, gt_heatmaps from your test set
# If not, get a sample like this:
sample_image, _, gt_heatmaps = test_dataset[3]
input_image = sample_image.unsqueeze(0).to(device)
with torch.no_grad():
    pred_heatmaps = model(input_image).cpu().squeeze(0)

input_img_np = sample_image.permute(1, 2, 0).numpy()
input_img_np = (input_img_np * 255).astype(np.uint8)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))

for i in range(4):
    gt_heatmap = gt_heatmaps[i].numpy()
    gt_heatmap_norm = (gt_heatmap - gt_heatmap.min()) / (np.ptp(gt_heatmap) + 1e-6)
    axes[0, i].imshow(gt_heatmap_norm, cmap='hot')
    axes[0, i].set_title(f"GT Heatmap {i+1}")
    axes[0, i].axis('off')

    pred_heatmap = pred_heatmaps[i].numpy()
    pred_heatmap_norm = (pred_heatmap - pred_heatmap.min()) / (np.ptp(pred_heatmap) + 1e-6)
    axes[1, i].imshow(pred_heatmap_norm, cmap='hot')
    axes[1, i].set_title(f"Pred Heatmap {i+1}")
    axes[1, i].axis('off')

    pred_heatmap_color = cm.jet(pred_heatmap_norm)[:, :, :3]
    heatmap_resized = cv2.resize(pred_heatmap_color, (input_img_np.shape[1], input_img_np.shape[0]))
    overlay = (0.6 * input_img_np / 255.0 + 0.4 * heatmap_resized)
    overlay = np.clip(overlay, 0, 1)
    axes[2, i].imshow(overlay)
    axes[2, i].set_title(f"Overlay {i+1}")
    axes[2, i].axis('off')

plt.suptitle("Row 1: Ground Truth Heatmaps | Row 2: Predicted Heatmaps | Row 3: Overlay on Input Image", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
def predict_points(image, heatmap_size=56, image_size=(224, 224)):
    """
    Plots the predicted and ground truth points on the image.
    """
    with torch.no_grad():
        pred_heatmaps = model(image).cpu().squeeze(0)
    
    pred_points_hm = heatmaps_to_points(pred_heatmaps)
    pred_points_img = points_to_image_coordinates(pred_points_hm, heatmap_size, image_size)
    
    return np.array(pred_points_img)

def calculate_homography_warped(image_np, pred_points, target_size):
    
    target_corners = np.array([
        [0, 0],
        [target_size - 1, 0],
        [target_size - 1, target_size - 1],
        [0, target_size - 1],
    ], dtype=np.float32)

    H, _ = cv2.findHomography(pred_points, target_corners)
    warped_image = cv2.warpPerspective(image_np, H, (target_size, target_size))

    return H, warped_image

In [ ]:
def plot_prediction_vs_ground_truth(image_np, gt_points_img, pred_points_img, distances, sample_idx):
    """
    Plots the predicted and ground truth points on the image.
    """
    _, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(image_np)

    gt_points_np = np.array(gt_points_img)
    ax.plot(gt_points_np[:, 0], gt_points_np[:, 1], 'go-', label='Ground Truth')

    pred_points_np = np.array(pred_points_img)
    ax.plot(pred_points_np[:, 0], pred_points_np[:, 1], 'ro-', label='Prediction')

    for j, txt in enumerate(['TL', 'TR', 'BR', 'BL']):
        ax.annotate(txt, (gt_points_np[j][0], gt_points_np[j][1]), color='green')
        ax.annotate(txt, (pred_points_np[j][0], pred_points_np[j][1]), color='red')

    ax.legend()
    ax.set_title(f"Sample {sample_idx+1} - Avg Dist: {np.mean(distances):.2f} px")
    plt.show()

def calculate_distance_pred_gt(pred_points_img, gt_points_img):
    """
    Calculates the distance between predictions and the ground truth points
    """
    distances = []
    for pred_pt, gt_pt in zip(pred_points_img, gt_points_img):
        dist = np.linalg.norm(np.array(pred_pt) - np.array(gt_pt))
        distances.append(dist)
    return distances

def plot_prediction_and_warped(image_np, gt_points_img, pred_points_img, distances, warped_image, sample_idx, show=True):
    """
    Plots the predicted and ground truth points on the image and the warped board.
    If show=False, returns the figure and axes objects without displaying.
    """
    _, axes = plt.subplots(1, 2, figsize=(12, 6))
    # Original image with points
    axes[0].imshow(image_np)
    gt_points_np = np.array(gt_points_img)
    axes[0].plot(gt_points_np[:, 0], gt_points_np[:, 1], 'go-', label='Ground Truth')
    pred_points_np = np.array(pred_points_img)
    axes[0].plot(pred_points_np[:, 0], pred_points_np[:, 1], 'ro-', label='Prediction')
    for j, txt in enumerate(['TL', 'TR', 'BR', 'BL']):
        axes[0].annotate(txt, (gt_points_np[j][0], gt_points_np[j][1]), color='green')
        axes[0].annotate(txt, (pred_points_np[j][0], pred_points_np[j][1]), color='red')
    axes[0].legend()
    axes[0].set_title(f"Sample {sample_idx+1} - Avg Dist: {np.mean(distances):.2f} px")
    axes[0].axis('off')
    # Warped board
    axes[1].imshow(warped_image)
    axes[1].set_title("Warped Board")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

To evaluate the model's performance we run the model to predict the chessboard corners on the test dataset and evaluate the distance between each predicted point and the ground truth point coordinates. In the end, we have managed to achieve an **average distance of <1 pixels**. In the following code, we select three images from the test dataset, use our trained model to predict the chessboard corners, and then apply a perspective transformation (warping) to align the board in the original image. This process allows us to visualize how accurately the model detects and rectifies the chessboard area for further analysis.

In [ ]:
def visualize_predictions(model, dataset, num_samples=5, image_size=224, heatmap_size=56, show=True):
    model.eval()
    total_distance = 0.0
    total_points = 0

    for i in range(len(dataset)):
        image, _, gt_heatmaps = dataset[i]
        input_image = image.unsqueeze(0).to(device)

        pred_points_img = predict_points(input_image)
        gt_points_img = points_to_image_coordinates(heatmaps_to_points(gt_heatmaps), heatmap_size, (image_size, image_size))

        distances = calculate_distance_pred_gt(pred_points_img, gt_points_img)
        total_distance += sum(distances)
        total_points += len(distances)

        image_np = image.permute(1, 2, 0).numpy()
        _, warped_image = calculate_homography_warped(image_np, pred_points_img, 640)

        if(show and num_samples > 0):
            plot_prediction_and_warped(
                image_np, gt_points_img, pred_points_img, distances, warped_image, i, show=show
            )
            num_samples -= 1

    average_distance = total_distance / total_points if total_points > 0 else 0.0
    return average_distance

average_dist = visualize_predictions(model, test_dataloader.dataset, num_samples=3, show=True)
print(f"Average point distance to ground truth: {average_dist:.2f} px")

# Integration Module

Currently, we already have the Yolo implemented, the model to detect the chessboard corners, but now it is necessary to group everything to fullfil the task provided in the project description.

In [ ]:
def read_image(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    return transform(image), image.size

This function is responsible to get the corners coordinates in the original image. This is what is being done:

- Read the image
- Predict the points (in *predict_points*)
- Scales the points (also in *predict_points*)

In [ ]:
def get_chessboard_corners(image_id, device=None, output_dir='yolo_chess_dataset'):
    """
    Get the four corners of a chessboard from an image by ID.
    Args:
        image_id: ID of the image to process (e.g., 'G000_IMG012')
        device: Device to run the model on ('cuda' or 'cpu')
        output_dir: Output directory with processed data
    Returns:
        numpy array of corner points in order [TL, TR, BR, BL]
    """
    # Construct image path
    image_path = os.path.join(output_dir, 'test', 'images', f"{image_id}.jpg")
    if not os.path.exists(image_path):
        # Try to find it in the images directory structure
        for folder in os.listdir('images'):
            candidate_path = os.path.join('images', folder, f"{image_id}.jpg")
            if os.path.exists(candidate_path):
                image_path = candidate_path
                break
        else:
            raise ValueError(f"Image {image_id} not found")
    
    # Load and preprocess the image
    image_tensor, (original_width, original_height) = read_image(image_path)
    input_image = image_tensor.unsqueeze(0).to(device)

    # Predicts the points in the heatmap and scales it to 224
    board_corners = predict_points(input_image, 56, (original_width, original_height))    
    return board_corners


#### Main function to run both yolo and corner detection

Here are some of the auxiliary functions to draw, to allow for a better visualization of the results

In [ ]:
def draw_chess_grid(image, target_size, color=(100, 100, 100), thickness=1):
    """Draws an 8x8 chess grid on the image."""
    square_size = target_size // 8
    for i in range(9):
        cv2.line(image, (0, i * square_size), (target_size, i * square_size), color, thickness)
        cv2.line(image, (i * square_size, 0), (i * square_size, target_size), color, thickness)
    return image

def draw_piece_circles(image, piece_positions, color=(255, 0, 0), radius=8):
    """Draws circles at the bottom center of each detected piece."""
    for piece in piece_positions:
        x, y = int(piece['position'][0]), int(piece['position'][1])
        cv2.circle(image, (x, y), radius, color, -1)
    return image

def draw_warped_pieces(image, pieces_by_type, target_size):
    """Draws circles and labels for each piece on the warped image."""
    for piece_type, positions in pieces_by_type.items():
        color_hash = hash(piece_type) % 255
        piece_color = (color_hash, 255 - color_hash, color_hash // 2)
        for pos in positions:
            x, y, conf = pos
            x, y = int(max(0, min(x, target_size-1))), int(max(0, min(y, target_size-1)))
            cv2.circle(image, (x, y), 8, piece_color, -1)
            cv2.putText(image, f"{piece_type} {conf:.2f}", (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, piece_color, 2)
    return image


def show_chessboard_detection_results(original_with_detections, warped, warped_with_pieces, pieces_dict):
    """
    Display the original image with detections, warped chessboard, and warped chessboard with detected pieces.
    Also prints the number of detected pieces by type.
    """
    _, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(original_with_detections)
    axes[0].set_title("Original with Detections")
    axes[0].axis('off')
    
    axes[1].imshow(warped)
    axes[1].set_title("Warped Chessboard")
    axes[1].axis('off')
    
    axes[2].imshow(warped_with_pieces)
    axes[2].set_title("Warped Chessboard with Detected Pieces")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\nDetected pieces by type:")
    for piece_type, positions in pieces_dict.items():
        print(f"{piece_type}: {len(positions)} pieces")

In [ ]:
def detect_and_warp_chessboard_with_pieces(image_id, corner_points, root_dir='', output_dir='yolo_chess_dataset', 
                                           model_path='runs/detect/chess_piece_detection/weights/best.pt', 
                                           target_size=800, conf_threshold=0.7):    
    # Use run_inference_by_id to get results and images
    results, original_img, _ = run_inference_by_id(
        image_id=image_id,
        root_dir=root_dir,
        output_dir=output_dir,
        model_path=model_path,
        conf_threshold=conf_threshold,
        debug=False
    )
    
    # Load categories
    categories = load_categories(root_dir)
    # Make a copy of the original image for detections
    original_with_detections = original_img.copy()
    
    H, warped_image = calculate_homography_warped(original_img, corner_points, target_size)
    
    # Create a copy for visualization
    warped_with_pieces = warped_image.copy()
    
    # Draw chess grid on warped image
    square_size = target_size // 8
    for i in range(9):
        cv2.line(warped_with_pieces, (0, i * square_size), (target_size, i * square_size), (100, 100, 100), 1)
        cv2.line(warped_with_pieces, (i * square_size, 0), (i * square_size, target_size), (100, 100, 100), 1)
    
    # Extract bottom centers of pieces from results
    original_piece_positions = []
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = box.conf.item()
            cls = int(box.cls.item())
            piece_name = categories.get(cls, 'unknown')
            
            # Calculate bottom center of the detected piece
            bottom_center_x = (x1 + x2) / 2
            bottom_center_y = y2  
            
            # Store position and metadata
            original_piece_positions.append({
                'name': piece_name,
                'position': (bottom_center_x, bottom_center_y),
                'confidence': conf,
                'bbox': (x1, y1, x2, y2)
            })
            
            cv2.circle(original_with_detections, (int(bottom_center_x), int(bottom_center_y)), 
                      8, (255, 0, 0), -1)
    
        pieces_by_type = {}
        for piece in original_piece_positions:
            try:
                piece_point = np.array([[[piece['position'][0], piece['position'][1]]]], dtype=np.float32)
                warped_point = cv2.perspectiveTransform(piece_point, H)
                wx, wy = warped_point[0, 0]
                if piece['name'] not in pieces_by_type:
                    pieces_by_type[piece['name']] = []
                pieces_by_type[piece['name']].append((wx, wy, piece['confidence']))
            except Exception as e:
                print(f"Error warping point {piece['position']}: {str(e)}")

        # Draw chess grid and pieces on warped image
        warped_with_pieces = draw_chess_grid(warped_image.copy(), target_size)
        warped_with_pieces = draw_warped_pieces(warped_with_pieces, pieces_by_type, target_size)

        return warped_image, warped_with_pieces, original_with_detections, pieces_by_type

In [ ]:
image_id = 'G000_IMG010' 
scaled_pred_points = get_chessboard_corners(image_id, output_dir='yolo_chess_dataset')

warped, warped_with_pieces, original_with_detections, pieces_dict = detect_and_warp_chessboard_with_pieces(
    image_id=image_id,
    corner_points=scaled_pred_points,
    root_dir='',
    output_dir='yolo_chess_dataset',
    target_size=800,
    conf_threshold=0.7
)

if warped is not None:
    show_chessboard_detection_results(original_with_detections, warped, warped_with_pieces, pieces_dict)

This function creates the dictionary that we create to later create the digital twin.

In [ ]:
def create_chessboard_matrix(pieces_dict, board_size=800):
    """
    Convert the detected chess pieces to an 8x8 chessboard matrix representation.
    
    Args:
        pieces_dict: Dictionary of pieces by type with their positions
        board_size: Size of the warped chessboard image
        
    Returns:
        8x8 matrix with piece representations and a mapping dictionary
    """
    board_matrix = [[None for _ in range(8)] for _ in range(8)]
    square_size = board_size // 8
    
    piece_mapping = {
        "white-pawn": "P", "white-knight": "N", "white-bishop": "B", 
        "white-rook": "R", "white-queen": "Q", "white-king": "K",
        "black-pawn": "p", "black-knight": "n", "black-bishop": "b", 
        "black-rook": "r", "black-queen": "q", "black-king": "k"
    }
    
    for piece_type, positions in pieces_dict.items():
        symbol = piece_mapping.get(piece_type, "?")  # Default to ? if piece type unknown
        
        for pos in positions:
            x, y, _ = pos
            col = int(x // square_size)
            row = int(y // square_size)
            if 0 <= row < 8 and 0 <= col < 8:
                board_matrix[row][col] = symbol
    
    return board_matrix

board = create_chessboard_matrix(pieces_dict, board_size=800)

#### Final Function - Create the digital twin from an image path

Finally, this function takes an image path and creates the digital twin integrating all the previous methods.

In [ ]:
unicode_pieces = {
    "K": u"\u2654", "Q": u"\u2655", "R": u"\u2656", "B": u"\u2657", "N": u"\u2658", "P": u"\u2659",
    "k": u"\u265A", "q": u"\u265B", "r": u"\u265C", "b": u"\u265D", "n": u"\u265E", "p": u"\u265F",
    None: ""
}

def plot_chessboard_unicode(board_matrix, ax=None):
    import matplotlib.pyplot as plt
    import numpy as np

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    for row in range(8):
        for col in range(8):
            color = '#f0d9b5' if (row + col) % 2 == 0 else '#b58863'
            rect = plt.Rectangle([col, 7-row], 1, 1, facecolor=color)
            ax.add_patch(rect)
            piece = board_matrix[row][col]
            if piece:
                ax.text(col + 0.5, 7-row + 0.5, unicode_pieces[piece], fontsize=36, ha='center', va='center')

    ax.set_xticks(np.arange(8) + 0.5)
    ax.set_yticks(np.arange(8) + 0.5)
    ax.set_xticklabels(['a','b','c','d','e','f','g','h'])
    ax.set_yticklabels(['1','2','3','4','5','6','7','8'][::-1])
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')

In [ ]:
def display_board_and_image(original_image_id):
    original_image_path = os.path.join('yolo_chess_dataset', 'test', 'images', f"{original_image_id}.jpg")
    if not os.path.exists(original_image_path):
        print(f"Error: Original image {original_image_path} not found")
        return
    
    original_image = cv2.imread(original_image_path)
    if original_image is None:
        print(f"Error: Could not load original image {original_image_path}")
        return
    
    original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    # Get corner points for the chessboard
    scaled_pred_points = get_chessboard_corners(
        image_id=original_image_id,
        output_dir='yolo_chess_dataset'
    )

    # Process image with detection and warping
    _, _, _, pieces_dict = detect_and_warp_chessboard_with_pieces(
        image_id=original_image_id,
        corner_points=scaled_pred_points,
        root_dir='',
        output_dir='yolo_chess_dataset',
        target_size=800,
        conf_threshold=0.7
    )

    # Create the board representation
    board_matrix = create_chessboard_matrix(pieces_dict, board_size=800)
    
    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    ax1.imshow(original_image_rgb)
    ax1.set_title(f"Original Image: {original_image_id}")
    ax1.axis('off')

    plot_chessboard_unicode(board_matrix, ax=ax2)
    ax2.set_title("Detected Chessboard Position")
    plt.tight_layout()
    plt.show()

# Example usage, with images that belong to test dataset
display_board_and_image('G000_IMG010')
display_board_and_image('G000_IMG001')
display_board_and_image('G033_IMG006')